In [1]:
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews

import requests
import json
warnings.filterwarnings('ignore')


# Get S&P500 companies' tickers

In [2]:
response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
companies_info_df = pd.read_html(response.content)[0]
companies_info_df = companies_info_df.drop(columns=["No.", "Stock Price", "% Change", "Revenue"])
companies_info_df.to_csv("Data/SPY_companies_info.csv", index=False)

In [3]:
# tech stocks, pharma stocks, oil stocks, tobacco stocks and Market
tickers = companies_info_df["Symbol"].to_list()

# News Data Download

## CapIQ Data
Data is acquired from SMU Library Website https://researchguides.smu.edu.sg/az.php?a=c

In [4]:
capiq_df = pd.read_excel("Data/capiq_news_data.xlsx")
capiq_df = capiq_df.rename(columns = {"Key Developments By Date": "date",
                            "Key Development Headline": "title",
                            "Key Development Sources": "source",
                            "Primary Industry": "topic"
                            })

capiq_df =  capiq_df.drop(columns=["Key Developments by Type", "Key Development Situation"])

In [5]:
capiq_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448274 entries, 0 to 448273
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   date                  448274 non-null  datetime64[ns]
 1   Company Name(s)       448274 non-null  object        
 2   title                 448274 non-null  object        
 3   source                448274 non-null  object        
 4   Business Description  448274 non-null  object        
 5   topic                 448274 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 20.5+ MB


In [6]:
# Get ticker
capiq_df["Ticker"] = capiq_df["Company Name(s)"].str.replace(r'[^(]*\(|\)[^)]*', '')
capiq_df["Ticker"] = capiq_df["Ticker"].str.split(':').str[-1]
capiq_df["Ticker"] = capiq_df["Ticker"].str.replace(r'[^a-zA-Z]+', '')
capiq_df["Company Name(s) - Cleaned"] = capiq_df["Company Name(s)"].str.split("(").str[0]

print(capiq_df.shape)
capiq_df

(448274, 8)


,date,Company Name(s),title,source,Business Description,topic,Ticker,Company Name(s) - Cleaned
0,2014-01-01,"Motorola Solutions, Inc. (NYSE:MSI)",Motorola Solutions to Provide IDF's Battlefiel...,Other,"Motorola Solutions, Inc. provides public safet...",Communications Equipment,MSI,"Motorola Solutions, Inc."
1,2014-01-01,TE Connectivity plc (NYSE:TEL),TE Connectivity Brings Advanced Mobile Service...,Business Wire,"TE Connectivity plc, together with its subsidi...",Electronic Manufacturing Services,TEL,TE Connectivity plc
2,2014-01-01,"Leidos Holdings, Inc. (NYSE:LDOS)",Leidos Holdings Receives Follow-On Contract fr...,Datamonitor NewsWire,"Leidos Holdings, Inc., together with its subsi...",Research and Consulting Services,LDOS,"Leidos Holdings, Inc."
3,2014-01-01,MarketAxess Holdings Inc. (NasdaqGS:MKTX),MarketAxess Holdings Inc.'s Equity Buyback ann...,Capital IQ Buybacks Database,"MarketAxess Holdings Inc., together with its s...",Financial Exchanges and Data,MKTX,MarketAxess Holdings Inc.
4,2014-01-01,Amgen Inc. (NasdaqGS:AMGN); UCB SA (ENXTBR:UCB),Amgen and UCB Announces Results from Phase 2 T...,PR Newswire,Amgen Inc. (NasdaqGS:AMGN)Amgen Inc. discovers...,Amgen Inc. (NasdaqGS:AMGN) (Biotechnology); UC...,UCB,Amgen Inc.
...,...,...,...,...,...,...,...,...
448269,2024-12-04,CSX Corporation (NasdaqGS:CSX),CSX Corporation Presents at UBS Global Industr...,PR Newswire; Business Wire; GlobeNewswire; Com...,"CSX Corporation, together with its subsidiarie...",Rail Transportation,CSX,CSX Corporation
448270,2024-12-04,UnitedHealth Group Incorporated (NYSE:UNH),UnitedHealth Group Incorporated - Analyst/Inve...,Business Wire,UnitedHealth Group Incorporated operates as a ...,Managed Health Care,UNH,UnitedHealth Group Incorporated
448271,2024-12-04,LyondellBasell Industries N.V. (NYSE:LYB),LyondellBasell Industries N.V. Presents at Gol...,PR Newswire; Business Wire; GlobeNewswire; Com...,LyondellBasell Industries N.V. operates as a c...,Commodity Chemicals,LYB,LyondellBasell Industries N.V.
448272,2024-12-04,Freeport-McMoRan Inc. (NYSE:FCX),Freeport-McMoRan Inc. Presents at Mines and Mo...,Company Website,Freeport-McMoRan Inc. engages in the mining of...,Copper,FCX,Freeport-McMoRan Inc.


In [7]:
capiq_df = capiq_df[capiq_df['Ticker'].isin(tickers)]
print(capiq_df.shape)


(427102, 8)


In [8]:
capiq_df2 = capiq_df.groupby(['Ticker']).first().drop_duplicates()
company_names = capiq_df.groupby(['Ticker']).first().drop_duplicates()["Company Name(s) - Cleaned"].tolist()
tickers_sorted = capiq_df.groupby(['Ticker']).first().drop_duplicates().index.tolist()

len(company_names)

498

## Pygoognews 
https://github.com/kotartemiy/pygooglenews


In [ ]:
# Looping over 500 tickers: this run will take a few hours
gn = GoogleNews(lang = 'en')

news_df = pd.DataFrame()
for i, company in tqdm(enumerate(company_names)):
    top = gn.search(company)
    entries = top["entries"]
    df_temp = clean_goog_news(entries)
    df_temp["date"] = df_temp["date"].apply(lambda d: pd.to_datetime(d, errors='coerce').date())
    df_temp = df_temp.dropna()
    df_temp["Ticker"] = tickers_sorted[i]
    news_df = pd.concat([news_df.copy(), df_temp.copy()])


In [ ]:
news_df

,date,title,source,Ticker
0,2024-12-14,Stifel Financial Corp Trims Stake in Agilent T...,https://news.google.com/rss/articles/CBMivgFBV...,A
1,2024-12-13,Is There Now An Opportunity In Agilent Technol...,https://news.google.com/rss/articles/CBMi5gFBV...,A
2,2024-12-13,Agilent Technologies (A) Announces Reorganizat...,https://news.google.com/rss/articles/CBMi9gFBV...,A
3,2024-12-13,"Metagenomics Market Report by Product, Technol...",https://news.google.com/rss/articles/CBMi7AJBV...,A
4,2024-12-13,Agilent Technologies Announces Operational Res...,https://news.google.com/rss/articles/CBMiswFBV...,A
...,...,...,...,...
95,2024-11-19,Zoetis Inc. (NYSE:ZTS) Shares Sold by Cantillo...,https://news.google.com/rss/articles/CBMixgFBV...,ZTS
96,2024-11-20,Zoetis Inc. (NYSE:ZTS) Holdings Lowered by Por...,https://news.google.com/rss/articles/CBMiswFBV...,ZTS
97,2024-11-26,Bank of Montreal Can Has $191.81 Million Holdi...,https://news.google.com/rss/articles/CBMitwFBV...,ZTS
98,2024-11-21,"First Horizon Advisors Inc. Sells 7,987 Shares...",https://news.google.com/rss/articles/CBMiugFBV...,ZTS


In [ ]:
news_df.to_csv("Data/goog_news.csv", index = False)

## General News 
- Newscatcher for general news not targetted at specific stocks: https://github.com/kotartemiy/newscatcher


In [ ]:
topics = ['tech', 'news', 'business', 'science', 'finance', 'food', 'politics', 'economics', 'travel', 'entertainment', 'music', 'sport', 'world']
all_general_news = pd.DataFrame()

for t in tqdm(topics):
    news_temp = clean_newscatcher_news(t, n=5)
    all_general_news = pd.concat([all_general_news, news_temp])



100%|██████████| 13/13 [01:56<00:00,  8.99s/it]


In [ ]:
all_general_news['title'] = all_general_news['title'].str.replace(r'[^a-zA-Z]+', ' ')
all_general_news['title'] = all_general_news['title'].str.strip()

all_general_news = all_general_news[all_general_news['title'] != '']

all_general_news["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
all_general_news.sort_values(by='date', inplace=True)
all_general_news = all_general_news.reset_index(drop=True)
all_general_news

,date,title,source,topic
0,2024-12-09T08:11:00.020-05:00,Housing Dec th Weekly Update Inventory down We...,http://www.calculatedriskblog.com/2024/12/hous...,business
1,2024-12-09T10:57:00.000-05:00,ICE Mortgage Monitor Refinance Activity Increa...,http://www.calculatedriskblog.com/2024/12/ice-...,business
2,2024-12-09T14:15:00.001-05:00,Leading Index for Commercial Real Estate Decre...,http://www.calculatedriskblog.com/2024/12/lead...,business
3,2024-12-09T20:51:00.011-05:00,Tuesday No major economic releases,http://www.calculatedriskblog.com/2024/12/tues...,business
4,2024-12-10T09:07:00.001-05:00,Part Current State of the Housing Market Overv...,http://www.calculatedriskblog.com/2024/12/part...,business
...,...,...,...,...
866,"Wed, 27 Nov 2024 00:00:00 EST",A Southwest road trip with your best friend do...,https://www.10best.com/interests/travel-tips/s...,travel
867,"Wed, 27 Nov 2024 00:26:28 +0000",Episode Tales of Coachella Kendrick Lamar s su...,https://www.loudandquiet.com/podcasts/episode-...,music
868,"Wed, 27 Nov 2024 17:24:36 +0000",After UCS Advocacy Million People Protected By...,https://www.ucsusa.org/node/15708,science
869,"Wed, 27 Nov 2024 18:50:47 +0000",Clean Energy and Environmental Justice Win in ...,https://www.ucsusa.org/node/15710,science


In [ ]:
all_general_news.to_csv("Data/general_news.csv", index = False)

## Merging CapIQ, PyGoogleNews and NewsCatcher

- CapIQ and PyGoogleNews will be concatenated based on the 'ticker' column
- CapIQ and NewsCatcher will be concatenated based on the 'topic' column 
    * NewsCatcher has less and defined topics for each news. Hence, 'glove-twitter-25' is used to evaluate which topic for each stock in CapIQ has the closest similarity to the topic in NewsCatcher
    * Note that 'glove-twitter-25' required both texts to have the same number of words to evaluate its similarity, hence 7 random non-NA rows of the first word of the topic for each stock in CapIQ will be evaluated against each topic found in NewsCatcher

In [67]:
news_df = pd.read_csv("Data/goog_news.csv")
all_general_news = pd.read_csv("Data/general_news.csv")

In [12]:
capiq_df_tickers = capiq_df["Ticker"].unique().tolist()

capiq_newscatcher_df = pd.DataFrame()
for t in tqdm(capiq_df_tickers):
    news_temp = capiq_df[capiq_df["Ticker"] == t].copy()
    
    match_topic_newscatcher = get_topic(news_temp["Business Description"].dropna()) # already sorted by ticker, just get the first non-NA topic
    curr_newscatcher = all_general_news[all_general_news["topic"] == match_topic_newscatcher].copy()
    curr_newscatcher["Ticker"] = t
    capiq_newscatcher_df = pd.concat([capiq_newscatcher_df, news_temp.drop(columns=["Company Name(s)", "Business Description"]).copy(), curr_newscatcher.copy()])
   

100%|██████████| 498/498 [01:10<00:00,  7.11it/s]


In [68]:
print(capiq_newscatcher_df.shape)
capiq_newscatcher_df

(453688, 6)


,date,title,source,topic,Ticker,Company Name(s) - Cleaned
0,2014-01-01 00:00:00,Motorola Solutions to Provide IDF's Battlefiel...,Other,Communications Equipment,MSI,"Motorola Solutions, Inc."
21,2014-01-02 00:00:00,"Motorola Solutions, Inc. (NYSE:MSI) acquired T...",Capital IQ Transaction Database,Chart Venture Partners (Asset Management and C...,MSI,Chart Venture Partners; Core Management II Cor...
1114,2014-01-16 00:00:00,"City of Charlotte, N.C. Works with Motorola So...",Business Wire,Communications Equipment,MSI,"Motorola Solutions, Inc."
1409,2014-01-21 00:00:00,"Motorola Solutions, Inc. to Report Q4, 2013 Re...",Business Wire,Communications Equipment,MSI,"Motorola Solutions, Inc."
1513,2014-01-22 00:00:00,"Motorola Solutions, Inc., Q4 2013 Earnings Cal...",Business Wire; Company Website,Communications Equipment,MSI,"Motorola Solutions, Inc."
...,...,...,...,...,...,...
838,"Wed, 11 Dec 2024 18:00:00 +0000",The benefits of Ozempic and its kin may extend...,https://www.sciencenews.org/article/benefits-o...,science,GEV,NaN
840,"Wed, 11 Dec 2024 18:53:14 +0000",The tsunami killed hundreds of thousands Are w...,https://www.sciencenews.org/article/2004-tsuna...,science,GEV,NaN
845,"Wed, 11 Dec 2024 23:40:00 +0000",expert reaction to study on rates of colorecta...,https://www.sciencemediacentre.org/expert-reac...,science,GEV,NaN
868,"Wed, 27 Nov 2024 17:24:36 +0000",After UCS Advocacy Million People Protected By...,https://www.ucsusa.org/node/15708,science,GEV,NaN


In [70]:
all_news_df = pd.concat([capiq_newscatcher_df, news_df])
all_news_df["date"] = all_news_df["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
all_news_df = all_news_df.dropna(subset=["date", "Ticker", "title"])
all_news_df.sort_values(by='date', inplace=True)
all_news_df = all_news_df.reset_index(drop=True)

all_news_df

,date,title,source,topic,Ticker,Company Name(s) - Cleaned
0,2008-02-01,Bell Company and DRA Advisors Announce Plans t...,https://news.google.com/rss/articles/CBMixwFBV...,NaN,UDR,NaN
1,2008-05-15,Boulder to be first “Smart Grid City” - POWER ...,https://news.google.com/rss/articles/CBMibkFVX...,NaN,XEL,NaN
2,2008-05-15,Howard Hughes: A revolutionary recluse - Las V...,https://news.google.com/rss/articles/CBMiekFVX...,NaN,UHS,NaN
3,2009-09-08,Xcel launches Boulder as first 'smart grid' ci...,https://news.google.com/rss/articles/CBMidEFVX...,NaN,XEL,NaN
4,2011-11-12,Calvert Renovation to Begin in January - Patch,https://news.google.com/rss/articles/CBMifEFVX...,NaN,UDR,NaN
...,...,...,...,...,...,...
498830,2024-12-14,December th COVID Update COVID in Wastewater I...,http://www.calculatedriskblog.com/2024/12/dece...,business,EBAY,NaN
498831,2024-12-14,"Insider Sell: Matthew Bilunas Sells 69,166 Sha...",https://news.google.com/rss/articles/CBMirgFBV...,NaN,BBY,NaN
498832,2024-12-14,"Matthew M. Bilunas Sells 69,166 Shares of Best...",https://news.google.com/rss/articles/CBMiwAFBV...,NaN,BBY,NaN
498833,2024-12-14,Stream It Or Skip It Heretic on VOD an Idea Dr...,https://decider.com/2024/12/13/hugh-grant-here...,entertainment,STZ,NaN


## Filter Irrelevant News
Any news different form more than 80% of the data is removed


In [71]:
all_news_cleaned_df = pd.DataFrame()

for ticker in tqdm(tickers_sorted):
    news_temp = all_news_df[all_news_df["Ticker"] == ticker].copy()
    news_temp = remove_irrelevant_news(news_temp.copy(), "title", threshold=0.1, threshold_size=0.8)

    all_news_cleaned_df = pd.concat([all_news_cleaned_df, news_temp.copy()])

all_news_cleaned_df = all_news_cleaned_df.reset_index(drop=True)

100%|██████████| 498/498 [01:27<00:00,  5.68it/s]


In [72]:
all_news_cleaned_df.shape

(458785, 6)

In [57]:
all_news_cleaned_df.to_csv("Data/all_news.csv", index=False)

## Removing Similar News


In [78]:
news_filtered = pd.DataFrame()

threshold = 0.75
set_interval = 15 # days
for ticker in tqdm(tickers_sorted):
    news_temp = all_news_cleaned_df[all_news_cleaned_df["Ticker"] == ticker].copy()
    all_test_dates = news_temp['date'].unique()
    news_filtered_temp = pd.DataFrame()
    for d in all_test_dates[:-set_interval]:
        date_threshold = datetime.strptime(np.datetime_as_string(d, unit='D'), "%Y-%m-%d") + timedelta(days=set_interval)
        date_threshold = date_threshold.strftime("%Y-%m-%d")
        test_temp2 = news_temp[(news_temp['date']>= d) & (news_temp['date'] <= date_threshold)].copy()
        test_temp2 = remove_similar_news(test_temp2, "title", threshold=threshold)
        news_filtered_temp = pd.concat([news_filtered_temp, test_temp2])
    news_filtered = pd.concat([news_filtered, news_filtered_temp.copy()])


news_filtered = news_filtered.drop_duplicates()
news_filtered = news_filtered.reset_index(drop=True)

100%|██████████| 498/498 [24:30<00:00,  2.95s/it] 


In [79]:
news_filtered["date"] = pd.to_datetime(news_filtered["date"], infer_datetime_format=True)
news_train_filtered = news_filtered[news_filtered["date"].dt.year < 2024].copy()
news_test_filtered = news_filtered[news_filtered["date"].dt.year == 2024].copy()

print(news_train_filtered.shape)
print(news_test_filtered.shape)

(349924, 6)
(67604, 6)


In [80]:
print(news_train_filtered.shape)
print(news_test_filtered.shape)

(349924, 6)
(67604, 6)


In [82]:
news_train_filtered.to_csv("Data/news_data_train.csv", index = False)
news_test_filtered.to_csv("Data/news_data_test.csv", index = False)

# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [77]:
stocks = yf.download(tickers, threads=True, group_by='ticker', start='2024-01-01', end='2024-12-16', multi_level_index=False)

[                       0%                       ]  2 of 503 completed

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-16)')
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


In [78]:
# Convert Stocks timezone to UTC 0
stocks.index = (
    stocks.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [79]:
stocks

Ticker           AMTM                                                     \
Price            Open   High        Low      Close  Adj Close     Volume   
Date                                                                       
2014-01-02        NaN    NaN        NaN        NaN        NaN        NaN   
2014-01-03        NaN    NaN        NaN        NaN        NaN        NaN   
2014-01-06        NaN    NaN        NaN        NaN        NaN        NaN   
2014-01-07        NaN    NaN        NaN        NaN        NaN        NaN   
2014-01-08        NaN    NaN        NaN        NaN        NaN        NaN   
...               ...    ...        ...        ...        ...        ...   
2024-12-09  23.650000  24.24  23.000000  23.750000  23.750000  2612300.0   
2024-12-10  23.799999  25.07  23.738001  24.020000  24.020000  2895000.0   
2024-12-11  24.440001  24.99  23.459999  23.850000  23.850000  2266800.0   
2024-12-12  23.590000  24.34  23.049999  23.610001  23.610001  1724700.0   
2024-12-13  23.639999  23.75  22.650000  23.049999  23.049999  1598900.0   

Ticker            TTWO                                      ...         LMT  \
Price             Open        High         Low       Close  ...         Low   
Date                                                        ...               
2014-01-02   17.270000   17.540001   17.150000   17.530001  ...  145.839996   
2014-01-03   17.500000   17.750000   17.500000   17.629999  ...  146.479996   
2014-01-06   17.670000   17.750000   17.420000   17.600000  ...  146.100006   
2014-01-07   17.639999   18.270000   17.520000   18.110001  ...  147.500000   
2014-01-08   17.950001   18.170000   17.730000   17.799999  ...  147.710007   
...                ...         ...         ...         ...  ...         ...   
2024-12-09  188.970001  191.020004  187.669998  187.899994  ...  508.549988   
2024-12-10  186.710007  188.240005  184.710007  185.360001  ...  508.100006   
2024-12-11  188.880005  191.259995  186.050003  190.460007  ...  503.309998   
2024-12-12  189.330002  190.539993  188.270004  189.639999  ...  488.709991   
2024-12-13  188.300003  188.649994  184.699997  185.479996  ...  492.179993   

Ticker                                              WAT              \
Price            Close   Adj Close   Volume        Open        High   
Date                                                                  
2014-01-02  146.070007  108.208145  1115900  100.000000  100.169998   
2014-01-03  147.059998  108.941452   843400   99.129997   99.779999   
2014-01-06  146.279999  108.363625  1134800   98.370003   98.870003   
2014-01-07  148.610001  110.089706  1684100   99.040001  100.519997   
2014-01-08  148.500000  110.008232  1222700  100.489998  101.360001   
...                ...         ...      ...         ...         ...   
2024-12-09  510.010010  510.010010  1170500  385.089996  392.679993   
2024-12-10  512.940002  512.940002  1178900  394.329987  395.760010   
2024-12-11  504.239990  504.239990  1308000  394.399994  397.019989   
2024-12-12  496.579987  496.579987  1862800  384.429993  386.690002   
2024-12-13  494.649994  494.649994   990100  382.220001  383.109985   

Ticker                                                   
Price              Low       Close   Adj Close   Volume  
Date                                                     
2014-01-02   99.000000   99.220001   99.220001   335600  
2014-01-03   97.900002   98.040001   98.040001  1197500  
2014-01-06   98.050003   98.389999   98.389999   551000  
2014-01-07   98.500000  100.379997  100.379997   514200  
2014-01-08  100.129997  100.480003  100.480003   595900  
...                ...         ...         ...      ...  
2024-12-09  383.790009  392.010010  392.010010   345200  
2024-12-10  385.410004  390.350006  390.350006   490100  
2024-12-11  384.399994  385.269989  385.269989   295400  
2024-12-12  381.410004  383.029999  383.029999   343400  
2024-12-13  373.410004  378.260010  378.260010   307900  

[2757 rows x 3018 co

In [80]:
market = yf.download(["SPY"], threads=True, group_by='ticker', start='2024-01-01', end='2024-12-16', multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [81]:
market.index = (
    market.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [82]:
market

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-01-02,183.979996,184.070007,182.479996,182.919998,151.242889,119636900
2014-01-03,183.229996,183.600006,182.630005,182.889999,151.218094,81390600
2014-01-06,183.490005,183.559998,182.080002,182.360001,150.779892,108028200
2014-01-07,183.089996,183.789993,182.949997,183.479996,151.705933,86144200
2014-01-08,183.449997,183.830002,182.889999,183.520004,151.738998,96582300
...,...,...,...,...,...,...
2024-12-09,607.690002,607.859985,604.080017,604.679993,604.679993,34742700
2024-12-10,605.369995,605.799988,602.130005,602.799988,602.799988,37234500
2024-12-11,605.780029,608.429993,605.500000,607.460022,607.460022,28677700


In [83]:
stocks.to_csv("Data/stocks_data.csv")

In [84]:
market.to_csv("Data/market_data.csv")